In [ ]:
# Importando as bibliotecas
import requests
import pandas as pd
import numpy as np
import re
import os


# * Listando os parâmetros que serão listados na API
parametros = {
    '@trimestre': "'20201'",
    '$top': 675,
    '$format': 'json',
    '$select': 'trimestre,UFTerminal,qtdTermPOS,qtdTermPOScompartilhados,qtdTermPOSchip,qtdTermPDV'
}

site = 'https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/INFRTERMDA(trimestre=@trimestre)'


# * Requisição + Tratamento de Erro
try:
    response = requests.get(url=site, params=parametros)
    response.raise_for_status()
    dados = response.json()
    
    # - Salvando os arquivos em um DataFrame
    dados_brutos = dados['value']
    df = pd.DataFrame(dados_brutos)

     # * Tratamento de Dados
    df_copia = df.copy()
    
    
    # Função que insere sublinhado antes de maiúsculas e converte para minúsculas
    def camel_to_snake(name):
        
        # Adiciona '_' antes de maiúsculas e remove espaços extras
        s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
        return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()

    # Aplicando a conversão em todas as colunas
    df_copia.columns = [camel_to_snake(col) for col in df.columns]
    

    # / Renomeando colunas
    df_copia = df_copia.rename(columns={
        'uf_terminal': 'estado',
        'qtd_term_pos': 'qtd_terminais_pos',
        'qtd_term_po_scompartilhados': 'qtd_terminais_pos_compartilhados',
        'qtd_term_po_schip': 'qtd_terminais_pos_com_chip',
        'qtd_term_pdv': 'qtd_terminais_pdv'
        })


    # / Separando o trimestre e o ano em colunas diferentes 
    df_copia['ano'] = df_copia['trimestre'].astype(str).str[-1]
    df_copia['trimestre'] = df_copia['trimestre'].astype(str).str[:4]
    
    # / Alterando o tipo do ano -> int
    df_copia['trimestre'] = df_copia['trimestre'].astype(int)
    df_copia['ano'] = df_copia['ano'].astype(int)
    
    # / Renomeando as colunas
    df_copia = df_copia.rename(columns={'trimestre': 'ano', 'ano': 'trimestre'})
    
    # / Reordenando as colunas
    coluna_trimestre = df_copia.pop('trimestre')
    df_copia.insert(1, 'trimestre', coluna_trimestre)
    
    
    # * 1. Mapeia qual é o mês e o dia final de cada número de trimestre
    fim_trimestre = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}
    
    # * 2. Junta o ano com o sufixo correspondente do trimestre
    df_copia['data_trimestre'] = (
        df_copia['ano'].astype(str) + df_copia['trimestre'].map(fim_trimestre)
    )
    # * 3. Converte para data
    df_copia['data_trimestre'] = pd.to_datetime(df_copia['data_trimestre']).dt.normalize()
    
    
    # / Reordenando coluna data_trimestre
    coluna_data_trimestre = df_copia.pop('data_trimestre')
    df_copia.pop('ano')
    df_copia.insert(0, 'data_trimestre', coluna_data_trimestre)



    display(df_copia.info())
    display(df_copia)


    # * Salvando os dados em um arquivo csv
    caminho_csv = os.path.join('..', 'data', 'stg_terminais_pos.csv')
    
    df_copia.to_csv(caminho_csv, index=False, sep=';', encoding='utf-8-sig')
    print(f'Arquivo salvo com sucesso em: {os.path.abspath(caminho_csv)}')
    
    
except requests.exceptions.RequestException as erro:
    print(f'Erro ao acessar a API: {erro}')



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 675 entries, 0 to 674
Data columns (total 7 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   data_trimestre                    675 non-null    datetime64[ns]
 1   trimestre                         675 non-null    int64         
 2   estado                            675 non-null    object        
 3   qtd_terminais_pos                 675 non-null    int64         
 4   qtd_terminais_pos_compartilhados  675 non-null    int64         
 5   qtd_terminais_pos_com_chip        675 non-null    int64         
 6   qtd_terminais_pdv                 675 non-null    int64         
dtypes: datetime64[ns](1), int64(5), object(1)
memory usage: 37.0+ KB


None

,data_trimestre,trimestre,estado,qtd_terminais_pos,qtd_terminais_pos_compartilhados,qtd_terminais_pos_com_chip,qtd_terminais_pdv
0,2020-03-31,1,RR,18122,11102,17868,1020
1,2020-03-31,1,GO,362852,230808,355486,23827
2,2020-03-31,1,TO,51434,29178,50387,2279
3,2020-03-31,1,MG,975679,631123,957566,43039
4,2020-03-31,1,AM,108521,66252,106226,5015
...,...,...,...,...,...,...,...
670,2026-03-31,1,PR,2705399,686319,1382201,188534
671,2026-03-31,1,MT,1062293,222914,504107,60176
672,2026-03-31,1,PE,2058387,591095,966194,33818
673,2026-03-31,1,SE,473173,128190,248943,4714


Arquivo salvo com sucesso em: c:\Users\mathe\OneDrive\Documentos\Meus Projetos\Análise de Dados\Projeto end-to-end\data\qtd_terminais_pos_pdv.csv
